# Word Translation Entropy (*CRITT server version*):

## [CRITT Academy](https://github.com/Critt-Kent/CRITT-Academy) Module 01: Lesson 07

### What You Will Do In This Lesson

Entropy-related values are computed on a word level for each study. They can be re-computes for several studies, increasing the accuracy of the values, or on a fraction of a study. 

1. Install and import all necessary Python libraries

2. Recompute entropy-related values on a word level for ST tables across several studies:
   - `HTra`: word translation Entropy
   - `HTraN`: normalized `HTra`
   - `InfT`: word translation information $log(1/P(\text{translation}))$
   - `InfS`: grouping probability: Probability to cluster ST words for the translation

3. Carry over the word-based values to the segment level to compute:
    - `HTraN`: average normalized `HTraN`
    - `HTot`: sum over word-based `HTra` values values
    - `InfT`: mean word-based `InfT` values
    - `InfS`: mean word-based `InfS` values

4. Carry over the word-based `HTra`, `HTraN`, `InfS`, and `InfT` to other kinds of units, e.g., `PUs`, `AUs` 

### First time working with a CRITT Academy code notebook?

If you haven't yet gone through the [CRITT Academy Module 01, Lesson 01, “Code Notebooks and the TPR_DB” (CRITT server version)](https://github.com/Critt-Kent/CRITT-Academy/blob/main/modules/01_Foundations/01_Notebooks_and_TPRDB_CRITTserver.ipynb), you should do that first (this will help you understand Step 1 much better).

## Step 1. Import all necessary Python libraries


In [ ]:
# Environment Setup
import sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

# Enforce a minimum Python version of 3.9+
if sys.version_info < (3, 9):
    raise RuntimeError(
        f"❌ Python 3.9 or higher is required. "
        f"You are currently running Python {sys.version_info.major}.{sys.version_info.minor}."
    )

REQUIRED = {
    "numpy": "2.4.0",
    "pandas": "3.0.0",
    "scipy": "1.17.0",
    "matplotlib": "3.10.0",
    "seaborn": "0.13.0",
    "tprdb-utilities": "0.7.0",
}

def _check_versions():
    outdated = []
    for pkg, min_ver in REQUIRED.items():
        try:
            installed = version(pkg)
            if Version(installed) < Version(min_ver):
                outdated.append(f"  {pkg}: installed {installed}, need >={min_ver}")
        except PackageNotFoundError:
            raise ImportError(f"{pkg} is not installed")
    return outdated

# Check if dependencies are missing and install them automatically
try:
    import numpy as np
    import pandas as pd
    import scipy.stats
    import matplotlib.pyplot as plt
    import seaborn as sns
    import tprdb_utilities

    outdated = _check_versions()
    if outdated:
        raise ImportError("Outdated packages:\n" + "\n".join(outdated))
    else:
        print("🤠 All core dependencies are already installed. You are ready to go! 🤘")
except ImportError:
    print("Missing dependencies detected. Installing required packages...")

    try:
        # The %pip magic ensures installation happens in the active Jupyter kernel
        %pip install "numpy>=2.4.0" "pandas>=3.0.0" "scipy>=1.17.0" "matplotlib>=3.10.0" "seaborn>=0.13.0" "tprdb-utilities>=0.7.0"

        print("\n🤠 Installation complete 🤘 If imports fail on the next cell, please restart the kernel.")
    except Exception as e:
        print(f"❌ An error occurred during installation: {e}")
        print("If using Google Colab, you may just have to restart your runtime now to use the newly installed packages.")

In [ ]:
# When working on the CRITT server: set the path to the tprdb_utilities directory:

import sys
sys.path.append(f"/data/tprdb-utilities/src")  # point to the folder that CONTAINS the utilities package

In [ ]:
# Now, import the libraries

try:
    # Standard Python library imports
    import glob
    import os
    import re

    import numpy as np
    import pandas as pd
    import scipy.stats
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns

    # TPR-DB utilities import
    from tprdb_utilities import fetch_TPRDB_tables, read_TPRDB_tables

    print("✅ All imports successful!")

except ImportError as e:
    print(f"❌ An error occurred during imports: {e}")
    print("Please ensure all dependencies are installed and the kernel is restarted if necessary.")

## Step 2: Read ST data from different studies into a DataFrame

When the studies have the same source text and identical language pairs, we can re-compute the entropy values based on the joint dataset. 


In [ ]:
# this is the path to the TPR-DB on the CRITT server
mothership = '/data/critt/tprdb/'

# these studies use (English) Multiling texts translation into Chinese
studiesList=["RUC17", "RUCMT17", "STML18bolt", "HNUJml", "STC17bolt", "ZHMT19", "MS12"]

# read Source Token tables (ST) from several different studies
STdf = read_TPRDB_tables(
    path=mothership,
    user='PUBLIC',
    studies=studiesList,
    extension="st",
    verbose=1,
)

In [ ]:
# check distribution of tasks in the dataframe

sns.countplot(STdf, x="Task")

### Recompute joint entropy values over all sessions


In [ ]:
from tprdb_utilities.transformer import ST_entropy_df  # import the function from module

# Recompute joint entropy values for all studies
ST = ST_entropy_df(STdf, Verbose = 1)

# see the heading of the dataftame with all colums
pd.set_option('display.max_columns', 500)
ST.head()

### Visualize entropy values

In [ ]:
# plot Histogram
ax4 = sns.catplot(x="Task",
                 y="HTraN",
                 kind="bar",
                 data=ST,
                 height=5,
                 aspect=2
                )

# add labels
ax4.set(ylabel='HTraN', xlabel='Task', title='Normalized joint Word Translation Entropy')
plt.show()

### Recompute entropy values per task

In [ ]:
# Recompute entropy values per Task
# -- task 'E' has too few occurrances

STdf_P = STdf[STdf['Task'] == 'P'] 
STdf_T = STdf[STdf['Task'] == 'T'] 
STdf_M = STdf[STdf['Task'] == 'MT'] 
STdf_S = STdf[STdf['Task'] == 'ST'] 

print("Post-editing sessions:")
STp = ST_entropy_df(STdf_P, Verbose = 1)

print("Translation sessions:")
STt = ST_entropy_df(STdf_T, Verbose = 1)

print("Machine Translation sessions:")
STm = ST_entropy_df(STdf_M, Verbose = 1)

print("Sight Translation sessions:")
STs = ST_entropy_df(STdf_S, Verbose = 1)


print(STp.shape, STt.shape, STm.shape, STs.shape)

### Visualize segment-based entropy values

In [ ]:
# concatenate dataframes of the individual tasks
STdf_2 = pd.concat([STm, STp, STt, STs])

# set font and figure size 
plt.rcParams["figure.figsize"] = (10,10)
matplotlib.rc('xtick', labelsize=20) 
matplotlib.rc('ytick', labelsize=20)
sns.set(font_scale = 2)

# plot Histogram
ax4 = sns.catplot(x="Task",
                 y="HTraN",
                 kind="bar",
                 data=STdf_2,
                 height=5,
                 aspect=2
                )

# add labels
ax4.set(ylabel='Normalized HTra', xlabel='Task', title='Normalized Word Translation Entropy')
plt.show()

## Step 3: Read SG data from different studies

In [ ]:
# read the session data into a DataFrame

extension="sg"

SGdf = read_TPRDB_tables(
    path=mothership,
    user='PUBLIC',
    studies=studiesList,
    extension=extension,
    verbose=1,
)

### Recompute joint entropy values per segment

In [ ]:
# Compute SG entropy values based on joint re-computed ST entropy 
from tprdb_utilities.transformer import SG_entropy_df  # import function from the module

# compute segment-based entropy based on word-based entropy values
SG = SG_entropy_df(SGdf, ST, Verbose = 1)

# see the heading of the dataftame with all colums
pd.set_option('display.max_columns', 500)
SG.head(2)

### Vizualize joint segment-based entropy values

In [ ]:
# check distribution of tasks in the dataframe

sns.countplot(SG, x="Task")

In [ ]:
# plot Histogram
ax4 = sns.catplot(x="Task",
                 y="HTraN",
                 kind="bar",
                 data=SG,
                 height=5,
                 aspect=2
                )

# add labels
ax4.set(ylabel='Segment-based HTraN', xlabel='Task', title='Normalized joint Word Translation Entropy')
plt.show()

### Recompute SG entropy values per Task

In [ ]:
# Recompute entropy values per Task
# -- task 'E' has too few occurrances

SG_P = SG[SG['Task'] == 'P'].copy() 
SG_T = SG[SG['Task'] == 'T'].copy() 
SG_M = SG[SG['Task'] == 'MT'].copy() 
SG_S = SG[SG['Task'] == 'ST'].copy() 

print("Post-editing sessions:")
SGp = SG_entropy_df(SG_P, STp, Verbose = 1)

print("Translation sessions:")
SGt = SG_entropy_df(SG_T, STt, Verbose = 1)

print("Machine Translation sessions:")
SGm = SG_entropy_df(SG_M, STm, Verbose = 1)

print("Sight Translation sessions:")
SGs = SG_entropy_df(SG_S, STs, Verbose = 1)


print(SGp.shape, SGt.shape, SGm.shape, SGs.shape)

In [ ]:
# concatenate dataframes of the individual tasks
SGdf_2 = pd.concat([SGm, SGp, SGt, SGs])

### Visualize task-adjusted entropy values per segment

In [ ]:
# set font and figure size 
plt.rcParams["figure.figsize"] = (10,10)
matplotlib.rc('xtick', labelsize=20) 
matplotlib.rc('ytick', labelsize=20)
sns.set(font_scale = 2)

# plot Histogram
ax4 = sns.catplot(x="Task",
                 y="HTraN",
                 kind="bar",
                 data=SGdf_2,
                 height=5,
                 aspect=2
                )

# add labels
ax4.set(ylabel='Normalized HTra', xlabel='Task', title='Normalized Word Translation Entropy')
plt.show()

In [ ]:
# set font and figure size 
plt.rcParams["figure.figsize"] = (10,10)
matplotlib.rc('xtick', labelsize=20) 
matplotlib.rc('ytick', labelsize=20)
sns.set(font_scale = 2)

# plot Histogram
ax4 = sns.catplot(x="Task",
                 y="InfT",
                 kind="bar",
                 data=SGdf_2,
                 height=5,
                 aspect=2
                )

# add labels
ax4.set(ylabel='Average InfT', xlabel='Task', title='Average Translation Information Value (InfT)')
plt.show()

In [ ]:
# set font and figure size 
plt.rcParams["figure.figsize"] = (10,10)
matplotlib.rc('xtick', labelsize=20) 
matplotlib.rc('ytick', labelsize=20)
sns.set(font_scale = 2)

# plot Histogram
ax = sns.catplot(x="Task",
                 y="InfS",
                 kind="bar",
                 data=SGdf_2,
                 height=5,
                 aspect=2
                )

# add labels
ax.set(ylabel='Average InfS', xlabel='Task', title='Average Source Information Value')
plt.show()

## Step 4: Translation entropy values for PUs

In [ ]:
# read the PU data for translation sessions


PUdf = read_TPRDB_tables(
    path=mothership,
    user='PUBLIC',
    studies=studiesList,
    extension="pu",
    verbose=1,
)

print("Number of rows and columns:", PUdf.shape)


### Compute average entropy values per PU from joint ST HTraN values

In [ ]:
# Compute entropy values for all PUs based on re-computed HTra values
from tprdb_utilities.transformer import DF_entropy_df  # import function from the module

PU = DF_entropy_df(PUdf, ST, Verbose = 1)


# see the heading of the dataftame with all colums
pd.set_option('display.max_columns', 500)
PU.head(2)

In [ ]:
# check distribution of tasks in the dataframe

ax = sns.countplot(PU, x="Task" )

# add labels
ax.set(ylabel='Number of PUs', xlabel='Task', title='Number of PUs per Task')
plt.show()

In [ ]:

# plot data
# set font and figure size 
plt.rcParams["figure.figsize"] = (10,10)
matplotlib.rc('xtick', labelsize=10) 
matplotlib.rc('ytick', labelsize=10)
sns.set(font_scale = 1)


ax4 = sns.catplot(x="Task",
                 y="HTraN",
                 kind="bar",
                 data=PU,
                 height=5,
                 aspect=2
                )

# add labels
ax4.set(ylabel='Average HTraN', xlabel='Task', title='Normalized Word Word Translation Entropy (HTraN) per PU')
plt.show()



### Recompute entropy values per task

In [ ]:
# Recompute entropy values per Task
# -- task 'E' and MT hava too few occurrances

PU_P = PU[PU['Task'] == 'P'].copy() 
PU_T = PU[PU['Task'] == 'T'].copy() 
PU_S = PU[PU['Task'] == 'ST'].copy() 

print("Post-editing sessions:")
PUp = DF_entropy_df(PU_P, STp, Verbose = 1)

print("Translation sessions:")
PUt = DF_entropy_df(PU_T, STt, Verbose = 1)

print("Sight Translation sessions:")
PUs = DF_entropy_df(PU_S, STs, Verbose = 1)


print(PUp.shape, PUt.shape, PUs.shape)

In [ ]:
# concatenate dataframes of the individual tasks
PUdf_2 = pd.concat([PUp, PUt, PUs])

In [ ]:
# set font and figure size 
plt.rcParams["figure.figsize"] = (10,10)
matplotlib.rc('xtick', labelsize=20) 
matplotlib.rc('ytick', labelsize=20)
sns.set(font_scale = 2)

# plot Histogram
ax = sns.catplot(x="Task",
                 y="HTraN",
                 kind="bar",
                 data=PUdf_2,
                 height=5,
                 aspect=2
                )

# add labels
ax.set(ylabel='Normalized HTra', xlabel='Task', title='Average Entropy per Task')
plt.show()

# Step 4: Helper functions

In [ ]:
# compute entropy values 
def ST_entropy_df(DF, Verbose = 1) :

    # associate text IDs with list of sessions
    STfiles = SortTextsInStudy_df(DF['Session'].unique())

    L = []
    # loop over text IDs
    for textID in STfiles:

        # store sntropy counts in HTRA 
        HTRA, STables, Error = ST_countValues_df(DF, STfiles[textID], Verbose = Verbose)

        if(Error) : 
            print(f"ERROR: different Source Texts {textID} : Entropy values cannot be added!")
            continue

            
        if(Verbose) : 
            cnt = 0
            if (1 in HTRA) : cnt = HTRA[1]['cnt']
                
            print(f"Source Texts {textID} with instances {cnt} ")
            
        # produce entropy columns for the entire study and each session
        SCcolumns = ST_entropy_values(HTRA)
    
        for session in STables:
            if(session not in SCcolumns):
                print("Warning ST_entropy:", session)
                continue

            # map into dataframe
            df = pd.DataFrame(SCcolumns[session]).T

            # if take out redundant columns
            cols = list(df.columns)
            if('SToken' in cols) : cols.remove('SToken')

            # delete double columns in STables that are already available
            df2 = STables[session].drop(columns=cols, errors='ignore')
            df2 = df2.drop(columns=[f"{c}_x" for c in cols], errors='ignore')
            df2 = df2.drop(columns=[f"{c}_y" for c in cols], errors='ignore')

            # re-assign Id 
            df['Id'] = range(1, len(df) +1)

            # and merge
            L.append(pd.merge(df2, df, on=['Id', 'SToken'], how='inner'))

    return pd.concat(L)

# return a dictionary {text : [list_of_sessions] ... } 
def SortTextsInStudy_df(sessions) :
    """
    sessions: list of sessions
    
    """
        
    # store the list of sessions per source textId 
    # e.g.: Study[1][session1, session5 ...]
    Study = {} 

    # find how many versions per ST 
    for fn in sessions:

        session = os.path.basename(fn)
        
        # parse the filename
        match = re.search(r'_[^\d]+(\d+)', session)
        if(match) :
            # convert to int, text could be 1 or 01
            tid = int(match.group(1))
            Study.setdefault(tid, [])      
            # parse the filename 
            Study[tid].append(session)
        else:
            print("TextIdsInStudy:filename:", fn)
        
    return Study



# per ST word count values for translation, TT group, ST group 
def ST_countValues_df(DF, StudyTextID, Verbose = 0) :
    
    STable = {} # dictionary of sessions with dataframe
    HTRA = {}  # dictionary of text IDs with item counts
    ERR = {}
    for session in sorted(StudyTextID) :

        # pull out session and count frequency of items
        STable[session] = DF[DF['Session'] == session]
        
        for t in STable[session].itertuples() :
            HTRA.setdefault(t.Id, {})
            
            # check whether ST words are identical across texts
            if('ST' in HTRA[t.Id] and HTRA[t.Id]['ST'] != t.SToken) :
                if(Verbose > 2) : print(f"ERROR: Different ST texts: {fn}: seg:{t.STseg} STid:{t.STid} SToken:{t.SToken:<10}\t{HTRA[t.Id]['ST']} ")
                ERR.setdefault(session, set())
                ERR[session] = ERR[session].union({t.STseg}) 
            HTRA[t.Id]['ST'] = t.SToken
            HTRA[t.Id].setdefault('cnt', 0)
            HTRA[t.Id]['cnt'] += 1
            
            # Target Group (Translation) count
            HTRA[t.Id].setdefault('T', {})
            HTRA[t.Id]['T'].setdefault(t.TGroup, {})
            HTRA[t.Id]['T'][t.TGroup].setdefault('cnt', 0)
            HTRA[t.Id]['T'][t.TGroup]['cnt'] += 1
            HTRA[t.Id]['T'][t.TGroup].setdefault('sess', [])
            HTRA[t.Id]['T'][t.TGroup]['sess'].append(session)

            # Source Group count
            HTRA[t.Id].setdefault('S', {})
            HTRA[t.Id]['S'].setdefault(t.SGroup, {})
            HTRA[t.Id]['S'][t.SGroup].setdefault('cnt', 0)
            HTRA[t.Id]['S'][t.SGroup]['cnt'] += 1
            HTRA[t.Id]['S'][t.SGroup].setdefault('sess', [])
            HTRA[t.Id]['S'][t.SGroup]['sess'].append(session)

            #  Cross
            HTRA[t.Id].setdefault('C', {})
            HTRA[t.Id]['C'].setdefault(t.Cross, {})
            HTRA[t.Id]['C'][t.Cross].setdefault('cnt', 0)
            HTRA[t.Id]['C'][t.Cross]['cnt'] += 1
            HTRA[t.Id]['C'][t.Cross].setdefault('sess', [])
            HTRA[t.Id]['C'][t.Cross]['sess'].append(session)

            # Joint Source, Target, Cross
            STC = f"{t.SGroup}@@{t.TGroup}@@{t.Cross}"
            HTRA[t.Id].setdefault('STC', {})
            HTRA[t.Id]['STC'].setdefault(STC, {})
            HTRA[t.Id]['STC'][STC].setdefault('cnt', 0)
            HTRA[t.Id]['STC'][STC]['cnt'] += 1
            HTRA[t.Id]['STC'][STC].setdefault('sess', [])
            HTRA[t.Id]['STC'][STC]['sess'].append(session)

    for session in ERR:
        print(f"ERROR: different ST in: {session} : seg:{ERR[session]}")
        
    return (HTRA, STable, len(ERR) != 0)

def ST_entropy_values(HTRA) :
    
    # compute information values and entropy
    for tId in HTRA:
        
        # loop over every word
        for item in HTRA[tId]['T']:
            HTRA[tId]['T'][item]['P'] = HTRA[tId]['T'][item]['cnt'] / HTRA[tId]['cnt']
            HTRA[tId]['T'][item]['I'] = np.log(1/HTRA[tId]['T'][item]['P']) 
            HTRA[tId].setdefault('HTra', 0)
            # Target-group Entropy 
            HTRA[tId]['HTra'] += HTRA[tId]['T'][item]['P']  *  HTRA[tId]['T'][item]['I']
            
        for item in HTRA[tId]['S']:
            HTRA[tId]['S'][item]['P'] = HTRA[tId]['S'][item]['cnt'] / HTRA[tId]['cnt']
            HTRA[tId]['S'][item]['I'] = np.log(1/HTRA[tId]['S'][item]['P']) 
            HTRA[tId].setdefault('HSgrp', 0)
            # Source-group Entropy 
            HTRA[tId]['HSgrp'] += HTRA[tId]['S'][item]['P']  *  HTRA[tId]['S'][item]['I']
        
            
        for item in HTRA[tId]['C']:
            HTRA[tId]['C'][item]['P'] = HTRA[tId]['C'][item]['cnt'] / HTRA[tId]['cnt']
            HTRA[tId]['C'][item]['I'] = np.log(1/HTRA[tId]['C'][item]['P']) 
            HTRA[tId].setdefault('HCross', 0)
            # Cross Entropy 
            HTRA[tId]['HCross'] += HTRA[tId]['C'][item]['P']  *  HTRA[tId]['C'][item]['I']
            
        for item in HTRA[tId]['STC']:
            HTRA[tId]['STC'][item]['P'] = HTRA[tId]['STC'][item]['cnt'] / HTRA[tId]['cnt']
            HTRA[tId]['STC'][item]['I'] = np.log(1/HTRA[tId]['STC'][item]['P']) 
            HTRA[tId].setdefault('HSTC', 0)
            # joint Sgroup, Tgroup, Cross  Entropy 
            HTRA[tId]['HSTC'] += HTRA[tId]['STC'][item]['P']  *  HTRA[tId]['STC'][item]['I']

    #AltT    CountT  ProbT   HTra    AltS    ProbS   HSgrp   AltC    ProbC   HCross  AltSTC  ProbSTC HSTC
    # mapping type - entropy attribute
    E = {'T':'HTra', 'S':'HSgrp', 'C':'HCross', 'STC':'HSTC'}
    M = {}
    for tId in HTRA:
        SToken = HTRA[tId]["ST"]
        
        for tpe in E.keys(): 

            for item in HTRA[tId][tpe]:
                for s in HTRA[tId][tpe][item]['sess'] :
                    M.setdefault(s, {})
                    M[s].setdefault(tId, {})
                    M[s][tId]["SToken"] = SToken
                    M[s][tId][f"Count"] = HTRA[tId][tpe][item]['cnt'] 
                    M[s][tId][f"Alt{tpe}"] = len(HTRA[tId][tpe].keys())
                    M[s][tId][f"Prob{tpe}"] = HTRA[tId][tpe][item]['P']
                    M[s][tId][f"Inf{tpe}"] = HTRA[tId][tpe][item]['I']
                    M[s][tId][E[tpe]] = HTRA[tId][E[tpe]]
                    # normalized Entropy
                    M[s][tId][f"{E[tpe]}N"] = HTRA[tId][E[tpe]] *np.log(2)/np.log(HTRA[tId]['cnt']) 
                    
    return M

# Compute entopy for segments
def SG_entropy_df(SG, ST, Verbose = 1) :

    # make sure STseg is of same type
    ST['STseg'] = ST['STseg'].astype(str)
    SG['STseg'] = SG['STseg'].astype(str)

    # initialize entropy values
    SG['HTot'] = 0.0
    SG['HTraN'] = 0.0
    SG['InfS'] = 0.0
    SG['InfT'] = 0.0
    
    # associate text IDs with list of sessions
    TextSessions = SortTextsInStudy_df(SG['Session'].unique())

    # loop over text IDs
    for textID in TextSessions:

        if(Verbose) : print(f"Text Id:{textID}\tTexts:{len(TextSessions[textID])}")
            
        # loop over sessions per text IDs
        for session in sorted(TextSessions[textID]) :

            if(Verbose > 2) : print(f"\t{session}")
                
            # pull out session and count frequency of items
            sg = SG[SG['Session'] == session]
            st = ST[ST['Session'] == session]
                
            # print("SG_entropy1", n, H, L)
            # loop through segments
            for seg in set(sg.STseg) :
                
                N = []
                # accumulate HTraN values for segment
                for s in seg.split('+') : N.extend(st[st.STseg == s]['HTraN'])                    
                # and compute mean HTraN
                SG.loc[SG.STseg == seg, 'HTraN'] = np.mean(N) 
        
                N = []
                # accumulate HTra values for segment
                for s in seg.split('+') : N.extend(st[st.STseg == s]['HTra'])
                # and compute sum HTra
                SG.loc[SG.STseg == seg, 'HTot'] = np.sum(N) 
                
                N = []
                # accumulate InfS values for segment
                for s in seg.split('+') : N.extend(st[st.STseg == s]['InfS'])
                # and compute sum InfS
                SG.loc[SG.STseg == seg, 'InfS'] = np.mean(N) 
                
                N = []
                # accumulate InfT values for segment
                for s in seg.split('+') : N.extend(st[st.STseg == s]['InfT'])
                # and compute sum InfT
                SG.loc[SG.STseg == seg, 'InfT'] = np.mean(N)
                
    return SG
        
# Compute entopy based on ST 
def DF_entropy_df(DF, ST, Verbose = 1) :

    if ('SGid' not in DF) :
        print(f"DF_entropy_df-1: no SGid in DF: {DF.head(3)}")
        return DF
              
    # initialize entropy values
    DF['HTot'] = 0.0
    DF['HTraN'] = 0.0
    DF['InfS'] = 0.0
    DF['InfT'] = 0.0
    
    # associate text IDs with list of sessions
    TextSessions = SortTextsInStudy_df(DF['Session'].unique())

    # loop over text IDs
    for textID in TextSessions:

        if(Verbose) : print(f"Text Id:{textID}\tTexts:{len(TextSessions[textID])}")
            
        # loop over sessions per text IDs
        for session in sorted(TextSessions[textID]) :

            if(Verbose > 2) : print(f"\t{session}")
                
            # pull out session and count frequency of items
            df = DF[DF['Session'] == session]
            st = ST[ST['Session'] == session]
                
            # print("DF_entropy1", n, H, L)
            # loop through segments
            for index, unit in df.iterrows() :

                # get SGid from unit
                SGid = str(unit['SGid'])

                # no Id or not aligned
                if (SGid == '---') or (SGid == '0'):
                    if(Verbose > 1) :print(f"DF_entropy_df: {unit['StudySession']}\tId:{unit['Id']} SGid: {SGid}")
                    continue
                            
                    
                N = []
                # accumulate HTraN values from STid for unit SGid 
                for s in SGid.split('+') : N.extend(st[st.STid == int(s)]['HTraN'])

                if(N == []) :
                    print(f"DF_entropy_df-HTraN: {unit['StudySession']}\tId:{unit['Id']} SGid:{SGid} N:{N}")
                    continue
                      
                # and compute mean HTraN
                DF.loc[index, 'HTraN'] = np.mean(N) 
    
                N = []
                # accumulate HTra values for segment
                for s in SGid.split('+') : N.extend(st[st.STid == int(s)]['HTra'])
                if(N == []) :
                    print(f"DF_entropy_df-HTra: {unit['StudySession']}\tId:{unit['Id']} SGid:{SGid} N:{N}")
                    continue
                    
                # and compute sum HTra
                DF.loc[index, 'HTot'] = np.sum(N) 
                
                N = []
                # accumulate InfS values for segment
                for s in SGid.split('+') : N.extend(st[st.STid == int(s)]['InfS'])
                if(N == []) :
                    print(f"DF_entropy_df-InfS: {unit['StudySession']}\tId:{unit['Id']} SGid:{SGid} N:{N}")
                    continue
                # and compute sum InfS
                DF.loc[index, 'InfS'] = np.mean(N) 
                
                N = []
                # accumulate InfT values for segment
                for s in SGid.split('+') : N.extend(st[st.STid == int(s)]['InfT'])
                if(N == []) :
                    print(f"DF_entropy_df-InfT: {unit['StudySession']}\tId:{unit['Id']} SGid:{SGid} N:{N}")
                    continue
                    
                # and compute sum InfT
                DF.loc[index, 'InfT'] = np.mean(N)
                
    return DF